# ProxyFuse post-hoc probe
Contract: docs/methods/proxy-fusion/design.md. No training, no official test.

In [ ]:
import json, subprocess, sys, shutil, hashlib
from pathlib import Path
import torch
print('torch', torch.__version__, flush=True)
assert torch.cuda.device_count()==2 and all(torch.cuda.get_device_name(i)=='Tesla T4' and torch.cuda.get_device_capability(i)==(7,5) for i in range(2))
print('gpus', [torch.cuda.get_device_name(i) for i in range(2)], flush=True)
found=[p.parent for p in Path('/kaggle/input').rglob('proxyfuse_manifest.json')]
assert len(found)==1, found
inp=found[0]; out=Path('/kaggle/working/proxyfuse'); out.mkdir(parents=True, exist_ok=True)
manifest=json.loads((inp/'proxyfuse_manifest.json').read_text())
assert hashlib.sha256((inp/'proxyfuse.py').read_bytes()).hexdigest()==manifest['proxyfuse.py']
shutil.copytree(inp, out/'inputs-copy-src', ignore=shutil.ignore_patterns('*.pt','*.npy','*.npz'))
r=subprocess.run([sys.executable, str(inp/'proxyfuse.py'), '--inputs', str(inp), '--out', str(out)], check=True)
print(json.dumps(json.loads((out/'proxyfuse_results.json').read_text())['pass_all_seeds']), flush=True)